# Chapter 11: Blockchain and Distributed Ledgers — A Mathematical Framework for Trustless Coordination

> *"A purely peer-to-peer version of electronic cash would allow online payments to be sent directly from one party to another without going through a financial institution."*
> — Satoshi Nakamoto, *Bitcoin: A Peer-to-Peer Electronic Cash System* (2008)

> *"Code is law — until the law says it isn't."*
> — Lawrence Lessig, paraphrased

## Learning Objectives

By the end of this chapter, you should be able to:

1. Define a cryptographic hash function formally and construct the Merkle tree data structure.
2. Specify the blockchain chain structure and prove the computational infeasibility of retroactive modification under honest-majority assumptions.
3. Analyze Proof of Work as a resource auction and Proof of Stake as a weighted cooperative voting game.
4. Define the Byzantine fault tolerance threshold and explain why it bounds all consensus mechanisms.
5. Model smart contracts as formal state machines and identify the limits of on-chain formal verification.
6. Evaluate token-weighted governance against the cooperative game theory axioms of Chapter 6.
7. Analyze Regen Network as an implementation of the Planetary Ledger concept.

---

## 11.1 Trust, Institutions, and the Coordination Problem

Every economic transaction rests on trust. For most of economic history, trust has been generated by institutions: states, banks, clearinghouses, and rating agencies that stand between parties and guarantee performance. The cost of institutional trust is significant — and the 2008 financial crisis demonstrated that institutional guarantees can fail precisely when they matter most [C:Ch.5].

Blockchain technology is, at its mathematical core, a mechanism for generating trust without a trusted third party — replacing institutional guarantees with cryptographic ones. The coordination problem it solves is specific: how can agents who do not trust each other maintain a shared record that no single party can unilaterally alter? This is the double-spend problem in digital cash and the Byzantine generals problem in distributed computing.

This chapter develops the mathematical foundations of blockchain with the rigor appropriate to this book, then asks: what can and cannot blockchain do for cooperative economic governance? The connections run forward to mutual credit systems [C:Ch.25], ecological accounting [C:Ch.20], and digital commons governance [C:Ch.39].

---

## 11.2 Blockchain as a Data Structure

### 11.2.1 Cryptographic Hash Functions

**Definition 11.1 (Cryptographic Hash Function).** A function $H: \{0,1\}^* \to \{0,1\}^n$ is a cryptographic hash function if it satisfies:

1. **Efficiency:** $H(x)$ computable in polynomial time in $|x|$.
2. **Pre-image resistance:** Given $y$, computationally infeasible to find $x$ with $H(x) = y$. For any PPT adversary $\mathcal{A}$: $\Pr[H(\mathcal{A}(y)) = y] \leq \text{negl}(n)$.
3. **Second pre-image resistance:** Given $x$, infeasible to find $x' \neq x$ with $H(x') = H(x)$.
4. **Collision resistance:** Infeasible to find any pair $(x, x')$, $x \neq x'$, with $H(x) = H(x')$.

SHA-256 maps arbitrary-length inputs to 256-bit outputs. Its security rests on the avalanche effect: flipping one bit in the input changes approximately half the output bits unpredictably, making systematic inversion computationally equivalent to exhaustive search over $2^{256}$ inputs.

**Proposition 11.1 (One-Way Property, Informal).** Under the random oracle model, finding a pre-image of a given SHA-256 hash requires in expectation $2^{256}$ evaluations. At $10^{18}$ evaluations per second (the Bitcoin network's 2023 hash rate), this would require approximately $3.7 \times 10^{59}$ years.

### 11.2.2 The Merkle Tree

**Definition 11.2 (Merkle Tree).** Over data items $\{d_1, \ldots, d_n\}$ ($n = 2^k$), the Merkle tree is constructed as:
- Leaf nodes: $\ell_i = H(d_i)$
- Internal nodes: $\text{node}_{i,j} = H(\text{node}_{i-1,2j-1} \| \text{node}_{i-1,2j})$
- Root: $r = \text{node}_{k,1}$

**Proposition 11.2 (Logarithmic Verification).** Given only the Merkle root $r$ and a proof path of $O(\log n)$ sibling hashes, any party can verify inclusion of $d_i$ in $O(\log n)$ time without knowing other data items.

The Merkle tree compresses $O(n)$ transaction data into a single 32-byte root hash while preserving logarithmic verifiability — enabling light clients (mobile wallets, IoT devices) to verify transactions using only block headers rather than full transaction histories.

### 11.2.3 The Chain Structure and Immutability

**Definition 11.3 (Blockchain).** A blockchain $\mathcal{B} = (B_0, B_1, \ldots)$ is a sequence of blocks where block $B_t$ contains header $H_t = (\text{prev\_hash}_t, \text{merkle\_root}_t, \text{nonce}_t, \text{timestamp}_t)$, transaction set $\mathcal{T}_t$, and block hash $\text{block\_hash}_t = H(H_t)$, with linking condition $\text{prev\_hash}_t = \text{block\_hash}_{t-1}$.

**Theorem 11.1 (Computational Immutability — Nakamoto, 2008).** Under the honest-majority assumption, the probability that an adversary controlling fraction $q < 1/2$ of network hash power successfully rewrites a block with $z$ confirmations is:

$$P(\text{rewrite}) = 1 - \Phi\left(z\sqrt{\frac{1-q}{q}}\right) \cdot \left(\frac{q}{1-q}\right)^z$$

For $q = 0.30$ and $z = 6$: $P \approx 0.0015$. For $z = 12$: $P \approx 10^{-7}$.

*Proof sketch.* Model the race between the honest chain (rate $1-q$) and the adversarial chain (rate $q$) as a random walk with drift $1 - 2q > 0$. The adversary must close a gap of $z$ blocks; the probability follows from the negative binomial distribution of the random walk hitting zero from position $z$. $\square$

**Economic interpretation.** Six confirmations gives an adversary with 30\% of hash rate only a 0.15\% chance of success. The cost of a 51\% attack (renting sufficient hash power) exceeded the expected gain for all Bitcoin transactions in 2023, making the blockchain economically secure against well-resourced adversaries.

---

## 11.3 Consensus Mechanisms: Game Theory Under the Hood

### 11.3.1 Proof of Work as a Resource Auction

Miners compete to find nonce $\nu$ satisfying $H(\text{header} \| \nu) \leq \text{target}$. Finding a valid nonce requires in expectation $\text{difficulty} = 2^{256}/\text{target}$ hash evaluations. The protocol adjusts difficulty every 2016 blocks to maintain a 10-minute average block time.

**The mining game.** With $n$ miners each choosing hash power $h_i$ at cost $c$ per hash, and block reward $R$:

$$\pi_i = \frac{h_i}{H} \cdot R - c \cdot h_i, \quad H = \sum_j h_j$$

**Proposition 11.3 (Mining Nash Equilibrium).** In the symmetric case, the Nash equilibrium satisfies:

$$h_i^* = \frac{(n-1)R}{n^2 c}, \quad H^* = \frac{(n-1)R}{nc}, \quad \pi_i^* = \frac{R}{n^2}$$

*Proof.* FOC for miner $i$: $\partial\pi_i/\partial h_i = R \cdot H_{-i}/H^2 - c = 0$. In symmetric equilibrium $H_{-i} = (n-1)h^*$, substituting and solving gives the result. $\square$

**The energy problem.** Total energy expenditure in equilibrium: $H^* \cdot c = (n-1)R/n \approx R$ for large $n$. Nearly the entire block reward is dissipated as heat. By 2023, the Bitcoin network consumed approximately 120 TWh/year — comparable to a mid-sized country — with no productive output beyond security. This structural inefficiency is not a design flaw but a mathematical consequence of PoW's security model: security requires that attack cost exceed attack benefit, and in equilibrium energy expenditure rises to consume approximately the full block reward.

### 11.3.2 Proof of Stake as a Weighted Cooperative Voting Game

PoS replaces computational work with economic stake. Validators lock cryptocurrency as collateral; block proposal probability is proportional to stake; misbehavior triggers slashing (confiscation of a fraction of stake).

**Definition 11.4 (Weighted Voting Game).** The PoS consensus mechanism is a weighted voting game $[T; s_1, s_2, \ldots, s_n]$ with threshold $T = S/2$ where $S = \sum_j s_j$:

$$v(S) = \mathbb{1}\left[\sum_{i \in S} s_i \geq T\right]$$

**Proposition 11.4 (Stake and Voting Power).** For large, well-distributed validator sets, the Shapley value (Banzhaf index) of validator $i$ converges to their stake fraction: $\phi_i \approx s_i/S$.

**Energy efficiency.** Ethereum's transition from PoW to PoS ("The Merge", September 2022) reduced network energy consumption by approximately 99.95% — from ~78 TWh/year to ~0.01 TWh/year — while maintaining equivalent security guarantees. For this book's ecological framework [C:Ch.22], PoS is the appropriate consensus mechanism: it achieves trustless coordination without thermodynamic waste incompatible with planetary boundaries.

### 11.3.3 Byzantine Fault Tolerance

**Definition 11.5 (Byzantine Fault).** A Byzantine fault is a failure mode in which a node sends conflicting or malicious information to different network partitions — the digital analogue of the Byzantine generals problem (Lamport, Shostak, Pease, 1982).

**Theorem 11.2 (BFT Security Threshold).** In any consensus protocol, consistent agreement is impossible if the fraction of Byzantine-faulty nodes exceeds:

$$f \geq \frac{n}{3}$$

*Proof sketch.* Partition $n$ nodes into three equal groups $A$, $B$, $C$. A Byzantine coordinator sends conflicting messages to each group. With $f = n/3$ Byzantine nodes colluding across groups, honest nodes in $A$ cannot distinguish their scenario from the one in which $C$ holds the correct information. Therefore no algorithm can guarantee correct decisions. $\square$

**Economic implication.** No blockchain is secure against adversaries controlling more than one-third of validators or hash power. This is a hard mathematical limit, not a design deficiency. For cooperative blockchain governance to be secure, the cooperative community must control at least two-thirds of the relevant resource — making blockchain a complement to social trust, not a substitute for it.

---

## 11.4 Smart Contracts as Formal Mechanisms

### 11.4.1 Smart Contracts as State Machines

**Definition 11.6 (Smart Contract).** A smart contract is a deterministic program $\mathcal{P} = (S, \Sigma, \delta, s_0, F)$ where $S$ is the state set, $\Sigma$ the input alphabet (transactions, function calls, oracle data), $\delta: S \times \Sigma \to S$ the deterministic transition function, $s_0$ the initial state, and $F \subseteq S$ the terminal states.

Every node executes every state transition identically, and the result is committed to the blockchain. The Ethereum Virtual Machine is Turing-complete in principle; in practice the gas limit per block bounds computational complexity and prevents infinite loops.

### 11.4.2 Incentive-Compatible Contract Design

**Definition 11.7 (Incentive-Compatible Smart Contract).** A contract $\mathcal{P}$ is incentive-compatible for action $a^*$ if for all $a \neq a^*$:

$$\mathbb{E}[U_a(\mathcal{P}(y(a^*)))] \geq \mathbb{E}[U_a(\mathcal{P}(y(a)))] - \text{effort}(a)$$

where $y(a)$ is the on-chain verifiable outcome under action $a$.

The key constraint: smart contracts can condition payments on any on-chain observable variable but cannot directly observe off-chain effort or quality. Oracle networks (Chainlink, Band Protocol) extend contract reach by bringing off-chain data on-chain, but introduce a new trust requirement: oracle honesty must itself be incentive-compatible.

**Proposition 11.5 (Oracle Incentive-Compatibility).** An oracle reporting off-chain data is incentive-compatible if:

$$\pi_{\text{slash}} > \frac{\text{benefit from manipulation}}{\Pr(\text{manipulation detected})}$$

For oracle networks with $k \geq 7$ independent reporters where deviation from the majority triggers 50\% stake slashing, $\Pr(\text{detected}) > 0.99$ and the condition is satisfied for all plausible manipulation values.

### 11.4.3 The Limits of Formal Verification

Smart contracts are programs, and programs can have bugs. High-profile exploits include the DAO hack (2016, \$60M), Poly Network hack (2021, \$611M), and Wormhole bridge hack (2022, \$320M) — all through unexpected input sequences that passed code review.

**Definition 11.8 (Principal Vulnerability Classes):** Re-entrancy (external call before state update), integer overflow (arithmetic wrap-around), front-running (miners inserting transactions ahead of pending ones to extract MEV), and oracle manipulation (flash-loan-driven price feed distortion).

Formal verification tools (Certora Prover, Echidna) can prove absence of specific vulnerability classes, but no tool can prove absence of all bugs in a Turing-complete program — the halting problem bounds what verification can achieve. The practical implication: smart contracts are suitable for automating well-specified, limited-scope rules (payment release, token distribution, vote tallying), but are inappropriate for open-ended governance requiring contextual judgment.

---

## 11.5 Token Governance and the Cooperative Ideal

### 11.5.1 Token-Weighted Voting vs. Cooperative Governance

Most blockchain governance systems allocate voting rights proportional to token holdings — one token, one vote — implementing the weighted voting game of Definition 11.4. This differs fundamentally from the Rochdale one-member-one-vote principle.

**The Governance Gini.** Define the governance Gini $G_{\text{gov}}$ as the Gini coefficient of Shapley values $\{\phi_i\}$ in the governance game. For token-weighted voting: $G_{\text{gov}} = G(\{t_i/T\})$ — the token wealth Gini. For the cooperative ideal: $G_{\text{gov}} = 0$.

Empirical analysis of major DeFi governance tokens (2022) finds Gini coefficients of 0.85–0.97. In practice, 5–15 addresses hold majority voting power in most deployed DAOs.

### 11.5.2 The Plutocracy Failure Mode

**Definition 11.9 (Governance Plutocracy).** Token governance is plutocratic if a coalition holding fraction $q > 0.5$ of tokens can unilaterally change protocol rules to benefit itself at the expense of other holders.

**Proposition 11.6 (Plutocracy Condition).** Token governance degenerates to plutocracy whenever the top 50\% of token holders (by balance) hold more than 50\% of total supply — a condition satisfied at Gini above approximately 0.45 for most empirical distributions. Since virtually all deployed DAOs have token Gini above 0.85, essentially all current token governance is formally plutocratic.

Token governance fails the cooperative game axioms of Chapter 6:
- **Symmetry:** Violated — token holdings differ by orders of magnitude.
- **Null player property:** Violated — small holders have near-zero Shapley value but are fully affected by governance outcomes.
- **Efficiency:** Satisfied.
- **Additivity:** Approximately satisfied for separable proposals.

### 11.5.3 Cooperative Blockchain Design Principles

Four principles bring token governance closer to the cooperative ideal:

1. **Capped token holdings:** Limit maximum balance per member, enforcing approximate symmetry.
2. **Contribution-based issuance:** Issue governance tokens proportional to verifiable contributions (labor, data, ecological stewardship) rather than capital purchase — implementing Shapley value logic on-chain.
3. **Quadratic voting:** Weight votes as $v_i = \sqrt{t_i}$ rather than $v_i = t_i$, reducing concentrated holders' influence. Lalley and Weyl (2018) prove this maximizes utilitarian welfare under independent valuations.
4. **Multi-stakeholder governance layers:** Separate economic, operational, and stewardship governance into distinct decision tiers — implementing the polycentric structure of Chapter 13 on-chain.

---

## 11.6 Mathematical Model: Blockchain Security Under Economic Attack

**Setup.** A cooperative uses PoS with $n$ validators holding stakes $\{s_i\}$, total stake $S$. An adversary aims to conduct a double-spend by controlling fraction $q$ of total stake.

**Definition 11.10 (Attack Profitability).** An attack is profitable if $V_{\text{double spend}} > C_{\text{attack}} = q \cdot S \cdot \text{price} \cdot (1 + p_d \cdot \sigma)$ where $\sigma$ is the slashing fraction and $p_d$ is detection probability.

**Proposition 11.7 (Economic Security Bound).** The cooperative must lock minimum stake:

$$S_{\min} = \frac{V^*}{q^* \cdot \text{price} \cdot (1 + p_d \cdot \sigma)}$$

For $q^* = 1/3$ (BFT threshold), $\sigma = 0.5$, $p_d = 0.9$: $S_{\min} \approx 2.07 V^*$.

The cooperative must lock at least twice the value being secured — a fundamental result connecting blockchain security to the value of the activity being protected. For small transactions, this is achieved cheaply; for large transactions, layer-2 solutions that aggregate multiple transactions under a single on-chain anchor are economically necessary.

---

## 11.7 Worked Example: Cooperative Supply Chain Smart Contract

We design a smart contract for a 50-member cooperative supply chain — manufacturer plus 49 suppliers — automating payment release conditional on delivery confirmation, quality certification, and governance approval.

### 11.7.1 State Machine Specification

```
STATES: PENDING → DELIVERED → CERTIFIED → APPROVED → PAID
                              ↘                         ↗
                           DISPUTED ─── arbitration ──→

TRANSITIONS:
  PENDING    → DELIVERED : manufacturer calls delivery_confirm()
                           condition: timestamp ≤ deadline
  DELIVERED  → CERTIFIED : oracle network calls certify()
                           condition: median quality_score ≥ threshold
  CERTIFIED  → APPROVED  : cooperative governance vote
                           condition: votes_for > 25 (simple majority of 50)
  APPROVED   → PAID      : supplier calls claim_payment()
                           effect: transfer(supplier, payment_amount)
  DELIVERED  → DISPUTED  : quality_score < threshold
  DISPUTED   → PAID/CANCELLED : arbitration_council.resolve()
```

### 11.7.2 Incentive-Compatibility Proof

**Manufacturer:** False delivery confirmation creates a permanent on-chain record; oracle fails to certify (no physical goods arrive); contract moves to DISPUTED with potential liability. Honest reporting dominates. $\square$

**Oracle (7 independent reporters, 50\% slashing for deviation from median):** By Proposition 11.5, incentive-compatible for all plausible manipulation values. $\square$

**Governance voters (50 members, one token each):** Each member holds Shapley value $1/50$ — insufficient to determine outcomes unilaterally. Cooperative social norms impose costs on dishonest voting exceeding any individual benefit. Honest assessment dominates. $\square$

### 11.7.3 Gas Cost Estimation

| Operation | Gas units | Cost at 30 gwei, ETH = \$2,000 |
|:---|:---|:---|
| Contract deployment | 850,000 | \$51.00 |
| delivery\_confirm() | 45,000 | \$2.70 |
| oracle.certify() × 7 | 210,000 | \$12.60 |
| Governance vote (50 members) | 75,000 | \$4.50 |
| claim\_payment() | 35,000 | \$2.10 |
| **Total per transaction** | **1,215,000** | **\$72.90** |

For a $\$10,000$ supply chain transaction, gas costs represent 0.73\% of transaction value — competitive with traditional trade finance fees (1–3\% for letters of credit). Below approximately $\$5,000$, gas costs represent significant overhead, motivating layer-2 deployment.

---

## 11.8 Case Study: Regen Network and the Planetary Ledger

### 11.8.1 The Protocol

Regen Network is a Cosmos-based blockchain designed for ecological accounting — tracking, verifying, and monetizing ecological regeneration. Launched 2021, it is the most complete existing implementation of the Planetary Ledger concept [C:Ch.20]: a distributed system tracking ecological state variables and enforcing the Regeneration Condition contractually.

Traditional carbon credit markets rely on slow, opaque certification bodies (Verra, Gold Standard) subject to methodological disputes and capture. Regen Network moves certification logic on-chain: ecological state protocols — formal algorithms mapping physical measurements to credit issuances — are published as auditable smart contracts. Any party can verify the methodology; issuances follow automatically from verified measurements.

### 11.8.2 Formal Analysis: The Ecological State Protocol

**Definition 11.11 (Ecological State Protocol, ESP).** An ESP is a smart contract $\mathcal{P}_{\text{eco}}: \mathbf{M} \to C$ mapping verified measurement vector $\mathbf{M}$ to credit issuance $C \geq 0$.

For soil carbon sequestration, the measurement vector includes SOC concentration, bulk density, sampling depth, and area. The ESP computes:

$$C = \max\left(0,\ \frac{(\text{SOC}_{\text{current}} - \text{SOC}_{\text{baseline}}) \cdot \rho_b \cdot d \cdot A}{1000}\right) \cdot p_f$$

where $\rho_b$ is bulk density, $d$ is depth, $A$ is area, and $p_f \in [0,1]$ is a permanence factor discounting re-release risk.

**Connection to the Stewardship Constraint.** The ESP maps directly to natural capital dynamics [C:Ch.17]:

$$N_{\text{soil}} = \text{SOC} \cdot \rho_b \cdot d \cdot A$$

A positive credit issuance $C > 0$ contractually certifies $\dot{N}_{\text{soil}} > 0$ — the Stewardship Constraint $\dot{N} \geq 0$ in on-chain form. This is the Planetary Ledger concept made operational: ecological state variables mapped to blockchain records, enabling contracts that condition economic transactions on verified ecological improvement.

### 11.8.3 Completeness and Verifiability Assessment

**Completeness.** Current Regen Network protocols cover: soil carbon, above-ground biomass, species diversity (bird surveys), stream health metrics. Absent: atmospheric methane flux, soil microbial diversity, pollinator abundance, hydrological cycle metrics, ocean acidification — all included in the Planetary Boundaries framework [C:Ch.17]. Current coverage captures approximately 30–40\% of the ecologically relevant state variables needed for a complete Planetary Ledger.

**Verifiability.** Regen Network uses three tiers: Level 1 (satellite remote sensing, fully automated, limited scope), Level 2 (community verification, peer review), Level 3 (third-party scientific audit — high credibility but slow and expensive). Level 3 reintroduces the bottleneck blockchain was supposed to eliminate; the gap between Level 1's automation and Level 3's credibility is the central verification challenge.

**Assessment.** Regen Network is a genuine proof-of-concept demonstrating that blockchain-based ecological accounting is technically feasible. Its limitations are measurement scope and verification cost, not protocol design — both will diminish as IoT sensors, satellite resolution, and oracle coverage improve. The most important achievement is institutional: publishing ecological state protocols as auditable smart contracts eliminates the black-box certification that made traditional carbon markets vulnerable to manipulation. The transparency of the protocol — anyone can verify the methodology — is a cooperative governance achievement as significant as the technical implementation.

---

## Chapter Summary

This chapter developed the mathematical foundations of blockchain and distributed ledger technology as a cooperative economic institution.

Cryptographic hash functions provide computationally binding commitments (SHA-256, pre-image resistance requires $2^{256}$ operations); the Merkle tree achieves $O(\log n)$ transaction verification; the chain structure makes retroactive modification computationally infeasible under the honest-majority assumption, proven formally by Nakamoto's random walk analysis.

Proof of Work is a winner-take-all resource auction whose Nash equilibrium dissipates nearly the full block reward as energy — a structural incompatibility with ecological principles that PoS eliminates at 0.05\% of the energy cost. Both are bounded by the Byzantine fault tolerance threshold of $n/3$: the maximum adversarial fraction any consensus mechanism can tolerate.

Smart contracts formalize principal-agent problems as deterministic state machines, automating rule execution without a trusted third party, but limited to on-chain observables and subject to implementation bugs that no current verification tool can fully eliminate.

Token-weighted governance systematically fails the symmetry and null-player axioms of cooperative game theory at empirically observed token concentration levels (Gini 0.85–0.97). Capped holdings, contribution-based issuance, quadratic voting, and multi-stakeholder governance layers bring token governance closer to the cooperative ideal.

Regen Network demonstrates that blockchain-based ecological accounting is technically feasible, covers approximately 30–40\% of relevant state variables, and achieves transparency in ecological certification that traditional certification bodies cannot match — an early but genuine implementation of the Planetary Ledger.

Part III now opens with Chapter 12: what network architectures sustain and amplify the cooperative advantages established in Part II, and which undermine them?

---

## Exercises

**11.1** Define a cryptographic hash function formally (Definition 11.1).
(a) At $10^{18}$ SHA-256 evaluations per second, how long does it take an adversary to find a pre-image with probability 0.5?
(b) Trace the Merkle verification path for leaf $d_7$ in a tree with 8 leaves. Which sibling hashes must the verifier know?
(c) For an adversary with $q = 0.25$ of Bitcoin hash rate, compute $P(\text{successful rewrite})$ for $z = 3$, $6$, and $12$ confirmations using Theorem 11.1.

**11.2** The PoW mining game (Proposition 11.3):
(a) Verify the Nash equilibrium for $n = 5$ symmetric miners, $R = 3.125$ BTC, $c = 10^{-9}$ USD/hash. Report $h_i^*$, $H^*$, $\pi_i^*$.
(b) Show aggregate miner profit $n\pi_i^* = R/n \to 0$ as $n \to \infty$. What does this imply for mining viability?
(c) Compare Ethereum PoS validator yield (~4\% APY) to the PoW profit per unit of capital. Which distributes rewards more equitably? Analyze using Shapley value reasoning.

**11.3** For the cooperative supply chain contract (Section 11.7):
(a) Does raising the governance threshold to supermajority (37/50) improve or worsen incentive-compatibility?
(b) Compute the Shapley value of each voter under simple majority and supermajority rules. How does the threshold shift governance power?
(c) Is supermajority voting more or less robust to strategic bloc formation? Use the blocking coalition concept [C:Ch.6].

**★ 11.4** Model Bitcoin mining as a cooperative game.
(a) Define characteristic function $v(S)$ as the expected fraction of block rewards earned by coalition $S$.
(b) Show $v(S) = \sum_{i \in S} h_i/H$. Is this game superadditive? Convex?
(c) Compute the Shapley value $\phi_i(v)$ for a pool with hash fraction $h_i/H$. How does it compare to independent mining revenue?
(d) Does the cooperative game framework predict the observed concentration (20–40\% pools)? What forces outside the model further incentivize pool formation?

**★ 11.5** Quadratic voting allocates weight $v_i = \sqrt{t_i}$.
(a) For 100 voters with Pareto-distributed token balances ($\alpha = 1.5$), compute the Gini of token balances and of quadratic voting weights. How much does quadratic voting reduce governance concentration?
(b) Sketch the Lalley-Weyl (2018) argument that quadratic voting maximizes utilitarian welfare under independent valuations proportional to $\sqrt{t_i}$.
(c) Identify two Sybil attack strategies against quadratic voting. How does proof-of-personhood address each?

**★★ 11.6** Design a DAO governance structure for a 200-member producer cooperative with three tiers: 100 workers (labor contribution), 50 investors (capital), 50 community stakeholders (affected community).

(a) Specify a Shapley-value-inspired token issuance rule for each tier. Prove it satisfies the efficiency and symmetry axioms within each tier.
(b) Design a multi-stakeholder voting mechanism — operational decisions (workers only), capital allocation (workers + investors, weighted), long-run governance (all three tiers, equal weight). Prove no single tier can unilaterally capture the DAO.
(c) Specify the governance mechanism as a pseudocode smart contract state machine. Identify three potential attack vectors and specify the on-chain conditions preventing each.
(d) Assess your governance structure against the eight Ostrom design principles [C:Ch.14]. Which principles does the on-chain implementation satisfy well? Which require off-chain institutional arrangements?

---

*Part III opens with Chapter 12: the formal analysis of how network structure — small-world, scale-free, and cooperative architectures — determines economic efficiency, inequality, and resilience across trade networks, financial systems, and supply chains.*